# 01 — Pool basics: parallel Python inside JupyterLite

This notebook runs CPU-bound Python **in parallel, in your browser**. The
Pyodide kernel executing these cells is itself a Web Worker; `pyodide_pool`
spawns a pool of **nested workers**, each with its own Pyodide interpreter,
and ships work to them with cloudpickle.

You will:

1. install the `pyodide_pool` wheel that ships with this site,
2. create a 4-worker pool,
3. `await pool.submit(...)` a CPU-bound prime count,
4. fan 8 chunks out across the pool and compare wall-clock time against a
   serial loop, with a bar chart of the timings.

> **Environment requirements** — this page must be **cross-origin isolated**
> (COOP/COEP headers; the bundled `npm run serve:lite` server sets them), and
> the browser must support **workers spawning nested workers** (current
> Chromium, Firefox, and Safari all do). Pool workers boot Pyodide from the
> jsDelivr CDN, so the first run needs network access.

The Pyodide kernel supports top-level `await`; this notebook uses it
throughout.

## Install the wheel

The site ships the driver-side Python package as a wheel under
`files/wheels/`. The path below is absolute because the kernel worker
resolves relative URLs against its own script URL, not against the notebook;
this site is served at the domain root (see `scripts/serve-lite.mjs`).

In [ ]:
import piplite

await piplite.install("pyodide_pool")

## Create the pool

`create_pool` imports the self-contained JS bundle
(`/files/assets/pyodide-pool.browser.js`) from inside the kernel worker,
starts a JS `PyodidePool` whose workers spawn as nested Web Workers, and
installs the returned pool as the package-wide default — so both
`pool.submit(...)` and module-level `pyodide_pool.submit(...)` work from
here on.

In [ ]:
import pyodide_pool

pool = await pyodide_pool.create_pool(pool_size=4)
print("pool ready with", pool.pool_size, "workers")

## The workload: counting primes by trial division

Pure Python and CPU-bound — exactly the kind of work that freezes a
single-threaded kernel. π(2,000,000) = 148,933 is the correctness check.

In [ ]:
RANGE_START, RANGE_END = 2, 2_000_000
CHUNK_COUNT = 8
EXPECTED_TOTAL = 148_933  # π(2_000_000)


def count_primes(lo, hi):
    count = 0
    for n in range(lo, hi):
        if n < 2:
            continue
        if n == 2:
            count += 1
            continue
        if n % 2 == 0:
            continue
        d = 3
        is_prime = True
        while d * d <= n:
            if n % d == 0:
                is_prime = False
                break
            d += 2
        if is_prime:
            count += 1
    return count


def make_chunks(start, end, count):
    width = -(-(end - start) // count)  # ceiling division
    return [(lo, min(lo + width, end)) for lo in range(start, end, width)]


chunks = make_chunks(RANGE_START, RANGE_END, CHUNK_COUNT)
chunks

## Submit one task

`pool.submit(fn, *args)` cloudpickles the call, runs it on a pool worker,
and returns an awaitable result. Functions defined in this notebook travel
**by value** — workers never import your code from anywhere. The first task
a worker receives also pays its one-time Pyodide boot (~1–2 s from the
CDN); after that the worker is warm.

In [ ]:
lo, hi = chunks[0]
count = await pool.submit(count_primes, lo, hi)
print(f"primes in [{lo:,}, {hi:,}): {count:,}")

## Warm all workers

The comparison below should measure compute, not interpreter boot, so make
every worker boot now: `pool_size` concurrent no-op tasks force the pool to
spin all of them up.

In [ ]:
import asyncio

await asyncio.gather(*(pool.submit(lambda i=i: i) for i in range(pool.pool_size)))
print("all", pool.pool_size, "workers warm")

## Serial baseline

The same 8 chunks in a plain loop, right here in the kernel — one
single-threaded WebAssembly interpreter doing all the work (expect several
seconds; the page stays responsive because the kernel is a worker, but the
kernel itself is blocked).

In [ ]:
import time

t0 = time.perf_counter()
serial_counts = [count_primes(lo, hi) for lo, hi in chunks]
serial_seconds = time.perf_counter() - t0

serial_total = sum(serial_counts)
assert serial_total == EXPECTED_TOTAL, serial_total
print(f"serial: {serial_seconds:.2f} s — {serial_total:,} primes")

## Parallel map across the pool

`asyncio.gather` over one `submit` per chunk — the pool runs up to
`pool_size` chunks at once while the kernel stays free.

In [ ]:
t0 = time.perf_counter()
parallel_counts = await asyncio.gather(
    *(pool.submit(count_primes, lo, hi) for lo, hi in chunks)
)
parallel_seconds = time.perf_counter() - t0

parallel_total = sum(parallel_counts)
assert parallel_total == EXPECTED_TOTAL, parallel_total
speedup = serial_seconds / parallel_seconds
print(
    f"parallel: {parallel_seconds:.2f} s — {parallel_total:,} primes "
    f"— {speedup:.2f}× speedup"
)

## Timings

(The first `matplotlib` import downloads the package into the kernel — give
it a few seconds.)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(
    ["serial (kernel)", f"parallel ({pool.pool_size} workers)"],
    [serial_seconds, parallel_seconds],
    color=["#b3b2ab", "#2a78d6"],
    width=0.55,
)
ax.bar_label(bars, fmt="%.2f s", padding=3)
ax.set_ylabel("wall-clock (s)")
ax.set_title(f"π(2,000,000) across {CHUNK_COUNT} chunks — {speedup:.2f}× speedup")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.grid(axis="y", color="#e3e2dc", linewidth=0.8)
ax.set_axisbelow(True)
plt.show()

## Success marker

The e2e smoke test asserts this exact line; it doubles as the notebook's
own bottom-line check.

In [ ]:
assert parallel_seconds < serial_seconds, (parallel_seconds, serial_seconds)
print("POOL_DEMO_OK", round(speedup, 2))